# D1 model results

**DTSC3602 Group 9**

Does a trained classifier over embedding similarity plus preference features beat plain cosine similarity at sorting job postings into **Good Fit / Partial Fit / No Fit**?

Four models are compared on the same held-out split:

| | |
|---|---|
| baseline: MiniLM cosine | thresholds set from training class proportions |
| baseline: TF-IDF cosine | same, on the bag-of-words baseline |
| logistic regression | all 16 features, interpretable coefficients |
| gradient boosting | all 16 features, native missing-value handling |

Run `python -m src.model --train` first to produce `artifacts/metrics.json`.

> **Read every number against the proxy.** The labels come from our own category taxonomy, not hiring outcomes. These metrics say how well the model reproduces *our definition* of fit.

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C, viz

viz.use_style()

metrics = json.loads(C.METRICS_JSON.read_text(encoding="utf-8"))
results = metrics["results"]

print(f"selected model : {metrics['selected']}")
print(f"beats baselines: {metrics['beats_baselines']}")
print(f"train pairs    : {metrics['n_train_pairs']:,}")
print(f"test pairs     : {metrics['n_test_pairs']:,}")

## 1. The headline comparison

Macro F1 weights all three classes equally, so a model cannot look good by only predicting the majority `No Fit` class.

In [ ]:
table = pd.DataFrame([
    {"model": r["name"], "macro F1": r["macro_f1"], "accuracy": r["accuracy"],
     "P@10": r["precision_at_10"], "NDCG@10": r["ndcg_at_10"], "MRR": r["mrr"]}
    for r in results
]).set_index("model")
table.round(3)

In [ ]:
# Two panels rather than one dual-axis chart: classification and ranking are
# different measures and do not belong on a shared scale.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
y = np.arange(len(table))

for ax, cols, title in [
    (axes[0], ["macro F1", "accuracy"], "Classification"),
    (axes[1], ["P@10", "NDCG@10", "MRR"], "Ranking"),
]:
    height = 0.8 / len(cols)
    for i, col in enumerate(cols):
        offset = (i - (len(cols) - 1) / 2) * height
        bars = ax.barh(y + offset, table[col], height=height * 0.88,
                       color=viz.CATEGORICAL[i], label=col)
        viz.bar_labels(ax, bars, fmt="%.2f")
    ax.set_yticks(y)
    ax.set_yticklabels(table.index)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.12)
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.xaxis.grid(True); ax.yaxis.grid(False)
    viz.despine(ax)

plt.tight_layout(); plt.show()

**Why both families of metric.** A swipe deck *is* a ranked list. A model could score well on macro F1 while still putting weak cards first, which is exactly the failure a user would notice. NDCG@10 and P@10 measure the thing the product actually does.

## 2. Where the selected model makes mistakes

In [ ]:
best = next(r for r in results if r["name"] == metrics["selected"])
cm = np.array(best["confusion_matrix"])
cm_pct = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5.6, 4.8))
im = ax.imshow(cm_pct, cmap=viz.SEQUENTIAL_CMAP, vmin=0, vmax=1)

ax.set_xticks(range(3), C.LABELS)
ax.set_yticks(range(3), C.LABELS)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title(f"Confusion matrix — {metrics['selected']}")
ax.grid(False)

for i in range(3):
    for j in range(3):
        # Ink flips on the dark end of the ramp so labels stay legible.
        ax.text(j, i, f"{cm_pct[i, j]:.0%}\n{cm[i, j]:,}", ha="center", va="center",
                color="#ffffff" if cm_pct[i, j] > 0.55 else viz.INK, fontsize=9)

plt.tight_layout(); plt.show()

print("Row-normalised: each row sums to 100% of that true class.")
print(f"Good Fit misread as No Fit: {cm_pct[2, 0]:.1%} — the costly error, since "
      "those postings never reach the deck.")

## 3. Per-class performance

`Partial Fit` is the hardest class by construction — it sits between the other two and is defined by a hand-authored adjacency list, so it inherits every judgement call in that table.

In [ ]:
per_class = pd.DataFrame({
    name: {k: best["per_class"][name][k] for k in ("precision", "recall", "f1-score", "support")}
    for name in C.LABELS
}).T

fig, ax = plt.subplots(figsize=(7.5, 3.8))
x = np.arange(len(C.LABELS))
for i, metric in enumerate(["precision", "recall", "f1-score"]):
    bars = ax.bar(x + (i - 1) * 0.26, per_class[metric], width=0.24,
                  color=viz.CATEGORICAL[i], label=metric)
    viz.bar_labels(ax, bars, fmt="%.2f")

ax.set_xticks(x, C.LABELS)
ax.set_ylim(0, 1.12)
ax.set_title("Per-class performance")
ax.legend(loc="upper right", ncols=3)
ax.xaxis.grid(False)
viz.despine(ax)
plt.tight_layout(); plt.show()

per_class.round(3)

## 4. What the model learned

Logistic coefficients are on standardised features, so magnitudes are directly comparable. A positive coefficient for the `Good Fit` class means more of that feature pushes a pair toward Good Fit.

If gradient boosting was selected instead, `coefficients` is `null` and this section is skipped.

In [ ]:
coefs = metrics.get("coefficients")

if not coefs:
    print(f"{metrics['selected']} has no linear coefficients — skipping.")
else:
    good = pd.Series(coefs["Good Fit"]).sort_values()
    good = good[good.abs() > 0.01]

    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    # Diverging: sign is the message, so the two poles get opposite hues.
    colors = ["#e34948" if v < 0 else viz.CATEGORICAL[0] for v in good.values]
    bars = ax.barh(good.index, good.values, color=colors, height=0.7)
    viz.bar_labels(ax, bars, fmt="%.2f")
    ax.axvline(0, color=viz.AXIS, linewidth=1)
    ax.set_xlabel("coefficient (standardised features)")
    ax.set_title("Drivers of a Good Fit prediction")
    ax.xaxis.grid(True); ax.yaxis.grid(False)
    viz.despine(ax)
    plt.tight_layout(); plt.show()

    print("Strongest positive:")
    print(good.tail(5).round(3).to_string())

## 5. Fairness — is the model evenly distributed across groups?

D1 commits to matches being spread evenly. This measures whether they are. We report the spread rather than correcting it: rescaling predictions to force parity would hide the imbalance without fixing its cause, which lies in the posting corpus and the taxonomy.

A category with few postings assigned to it (see EDA §3) has fewer chances to be matched, so a low Good-Fit rate here usually traces back to corpus composition, not to anything the classifier invented.

In [ ]:
fair = pd.DataFrame(metrics["fairness"]["by_resume_category"]).T.sort_values("good_rate")

fig, ax = plt.subplots(figsize=(7.5, 6))
bars = ax.barh(fair.index, fair.good_rate, color=viz.CATEGORICAL[0], height=0.72)
viz.bar_labels(ax, bars, fmt="%.2f")
mean_rate = fair.good_rate.mean()
ax.axvline(mean_rate, color=viz.CATEGORICAL[1], linestyle="--", linewidth=2,
           label=f"mean {mean_rate:.2f}")
ax.set_xlabel("predicted Good-Fit rate on held-out pairs")
ax.set_title("Good-Fit rate by resume category")
ax.set_xlim(0, max(fair.good_rate.max() * 1.2, 0.1))
ax.legend(loc="lower right")
ax.xaxis.grid(True); ax.yaxis.grid(False)
viz.despine(ax)
plt.tight_layout(); plt.show()

print(f"spread (max - min): {metrics['fairness']['good_rate_spread']:.3f}")
print("\nlowest five:")
print(fair.head(5).round(3).to_string())

### By posting state

Same question, grouped by the state the posting is in rather than the resume's category. D1 names location explicitly alongside name and writing style as a bias risk. States with fewer than 15 test pairs are dropped from the spread statistic (a rate over a handful of postings is noise, not evidence).

In [ ]:
by_state = metrics["fairness"].get("by_state")
state_spread = metrics["fairness"].get("state_good_rate_spread")

if not by_state or state_spread is None:
    print("Not enough states with >= 15 test pairs to report a spread.")
else:
    state_fair = pd.DataFrame(by_state).T.sort_values("good_rate")

    fig, ax = plt.subplots(figsize=(7.5, max(3.5, 0.28 * len(state_fair))))
    bars = ax.barh(state_fair.index, state_fair.good_rate, color=viz.CATEGORICAL[0], height=0.72)
    viz.bar_labels(ax, bars, fmt="%.2f")
    mean_state_rate = state_fair.good_rate.mean()
    ax.axvline(mean_state_rate, color=viz.CATEGORICAL[1], linestyle="--", linewidth=2,
               label=f"mean {mean_state_rate:.2f}")
    ax.set_xlabel("predicted Good-Fit rate on held-out pairs")
    ax.set_title(f"Good-Fit rate by posting state ({len(state_fair)} states, n>=15)")
    ax.set_xlim(0, max(state_fair.good_rate.max() * 1.2, 0.1))
    ax.legend(loc="lower right")
    ax.xaxis.grid(True); ax.yaxis.grid(False)
    viz.despine(ax)
    plt.tight_layout(); plt.show()

    print(f"spread (max - min): {state_spread:.3f}")
    print(f"category spread was: {metrics['fairness']['good_rate_spread']:.3f}")
    print("
lowest five states:")
    print(state_fair.head(5).round(3).to_string())

## 6. Engagement data from the swipe demo

This is the instrument for D1's research question. It only has rows once someone has used the app — `streamlit run app/streamlit_app.py`.

The useful comparison for the final write-up is **like rate by predicted fit tier**: if the model is working, users should mark Good Fit cards as interesting more often than No Fit ones. Swipes under 1.5s are flagged as blind swipes and excluded, which is D1's data-quality safeguard.

In [ ]:
if not C.SWIPE_LOG.exists():
    print("No swipes logged yet — run the Streamlit app and swipe through a deck.")
else:
    log = pd.read_csv(C.SWIPE_LOG)
    clean = log[log.blind_swipe == 0]
    print(f"{len(log):,} swipes across {log.session_id.nunique()} session(s); "
          f"{len(log) - len(clean):,} flagged as blind")

    if len(clean):
        by_tier = (clean.groupby("fit_tier")
                        .agg(cards=("direction", "size"),
                             like_rate=("direction", lambda s: (s == "like").mean())))
        by_tier = by_tier.reindex([t for t in C.LABELS if t in by_tier.index])

        fig, ax = plt.subplots(figsize=(6, 3.6))
        bars = ax.bar(by_tier.index, by_tier.like_rate, width=0.6,
                      color=[viz.TIER_COLORS[t] for t in by_tier.index])
        viz.bar_labels(ax, bars, fmt="%.2f")
        ax.set_ylim(0, 1.12)
        ax.set_ylabel("like rate")
        ax.set_title("Like rate by predicted fit tier (blind swipes excluded)")
        ax.xaxis.grid(False)
        viz.despine(ax)
        plt.tight_layout(); plt.show()

        print(by_tier.round(3).to_string())
        print(f"\nmedian dwell: {clean.dwell_ms.median()/1000:.1f}s")

## Takeaways

Fill these in from the numbers above once the run completes:

1. **Did the trained model beat cosine alone?** `metrics['beats_baselines']` answers this on both macro F1 and NDCG@10. If it is `False`, say so — a negative result about added complexity is a legitimate finding.
2. **Which features carried the signal?** §4 ranks them. If one feature dominates, the rest of the pipeline is decoration and should be justified or cut.
3. **How uneven is coverage across categories?** §5. Trace low-rate categories back to posting counts in EDA §3 before attributing the gap to the model.
4. **The headline caveat, repeated:** labels are a proxy built by us. The honest next step is a hand-labelled gold set of a few hundred pairs, scored by several group members so disagreement is visible.